# Pilot 4 · rank-1 GRPO arm

**Implemented: source preparation, sampling baseline and offline GRPO validation (tickets 01–03).** Training, GPU preflight, evaluation and measurement remain pending. This notebook does not launch GRPO training yet.

Pilot 2 trained rank-1 SFT on 512 GSM8K problems. Pilot 4 reuses those problems and optimizer prompt order, changing the objective to group-normalized reward training.

Sections 1–3 are CPU-only and verify saved Drive evidence. Sections 4–6 require one GPU and sample the untuned FP32 model. Section 8 uses tiny real models on CPU with no model download. It can run after dependencies are installed, without waiting for GPU sampling. FP32 for both arms is approved. Historical accuracy 0.640 is contextual, not a correctness gate.


In [ ]:
from pathlib import Path
import json, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
REPO_URL = 'https://github.com/mahadikprasad15/spar-pilot.git'
HARNESS_COMMIT = '6427e74a1637233cd6549817d8bb01c63cf65f4c'
REPO_DIR = Path('/content/spar-pilot')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/SPAR/spar-pilot/artifacts')
SFT_PLAN = 'pilot2-sft-fp32-l4-batch8-v1' # @param {type:"string"}
MEASUREMENT_PLAN = 'pilot3-inputs-v1' # @param {type:"string"}
PLAN_NAME = 'pilot4-grpo-fp32-v1' # @param {type:"string"}
SFT_CONFIG = ARTIFACT_ROOT / 'plans' / SFT_PLAN / 'sft.config.json'
MEASUREMENT_CONFIG = ARTIFACT_ROOT / 'plans' / MEASUREMENT_PLAN / 'activation.prepared.json'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
dirty = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True).strip()
assert not dirty, 'Preserve local repository edits before changing the checkout.'
subprocess.run(['git', 'fetch', 'origin', HARNESS_COMMIT], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'checkout', '--detach', HARNESS_COMMIT], cwd=REPO_DIR, check=True)
for source_path in [SFT_CONFIG, MEASUREMENT_CONFIG]:
    print(source_path, '| exists:', source_path.is_file())
    assert source_path.is_file(), 'Locate the saved source config in Drive before proceeding.'


## 2. Verify sources and prepare the plan

This calls the same public command tested offline. It verifies completed SFT checkpoints, frozen-base evidence, original training/evaluation data, actual saved prompt order and Pilot 3 token/mask hashes. It reuses those files rather than resampling or tokenizing again.

**Expected:** a prepared config, matching audit, metadata and completion marker. Repeating this cell revalidates/reuses the preparation. An incompatible source or corrupt artifact stops the command. Prepared source evidence does not mean training has run or the scientific protocol is frozen.


In [ ]:
subprocess.run([sys.executable, '-m', 'pilot_eval', 'grpo-prepare',
                '--sft-config', str(SFT_CONFIG), '--measurement-config', str(MEASUREMENT_CONFIG),
                '--name', PLAN_NAME, '--output-root', str(ARTIFACT_ROOT)],
               cwd=REPO_DIR, check=True)
PREPARED_PATH = ARTIFACT_ROOT / 'plans' / PLAN_NAME / 'grpo.prepared.json'


## 3. Read the matching audit

Review the identities and intended matches: 512 training problems, 150 held-out items, 150 GSM8K + 150 FineWeb measurement sequences, FP32 and rank-1 adapters. The saved SFT optimizer order is reused; the selected source indices alone do not specify training order.

The unresolved table is intentional. It identifies the scorer confirmation, exact verified loss, baseline subset/generation policy, final completion cap, monitor actions and preregistration. Later tickets resolve them before a full run. Historical aggregate results are not tests of this preparation.


In [ ]:
subprocess.run([sys.executable, '-m', 'pilot_eval', 'grpo-audit',
                '--config', str(PREPARED_PATH), '--output-root', str(ARTIFACT_ROOT)],
               cwd=REPO_DIR, check=True)
PLAN = json.loads(PREPARED_PATH.read_text())
print('Prepared config:', PREPARED_PATH)
print('Audit:', ARTIFACT_ROOT / PLAN['audit_path'])
print('Pending choices:', ', '.join(PLAN['pending']))
print('Next stage: untuned baseline sampling after its settings are explicitly resolved.')


## 4. Prepare the GPU sampling environment

Only sampling needs the GPU; source preparation above is CPU-only. Install the same pinned libraries used by the earlier pilots. If Colab requests a restart, restart and restore sections 1–3. This stage uses an untuned FP32 model on one GPU, not a hardware-specific T4/L4 code path.


In [ ]:
from pilot_eval.sft_backend import PINS
subprocess.run([sys.executable, '-m', 'pip', 'install', *[f'{k}=={v}' for k, v in PINS.items()]], check=True)


## 5. Supply explicit sampling choices

This is **128 problems from the 512 training problems**, eight draws each. It is neither the 150 held-out behavioural evaluation nor X1's deferred full 512-problem sampling baseline.

Create a JSON settings file in Drive. Required keys: `subset_ids` (128 unique training IDs in the chosen order), `scorer` (`gsm8k-flexible-v3`), `seed`, `groups_per_batch`, `max_new_tokens`, `temperature` (1.0), `top_p`, and `top_k`. No values are selected silently. One group is eight completions: `groups_per_batch=1` means an inference batch of eight, not one.

Choose the provisional cap, filters, batch policy and subset before running. The chosen sampling seed is distinct from cohort selection; record how you selected the subset in your notes. The command freezes all IDs/settings/batch seeds. Keep the same name/settings to resume; use a new name for a longer-cap extension.


In [ ]:
SAMPLING_SETTINGS_PATH = '' # @param {type:"string"}
BASELINE_NAME = 'pilot4-untuned-sampling-v1' # @param {type:"string"}
assert SAMPLING_SETTINGS_PATH.strip(), 'Supply the explicit reviewed settings JSON path first.'
SAMPLING_SETTINGS_PATH = Path(SAMPLING_SETTINGS_PATH)
assert SAMPLING_SETTINGS_PATH.is_file(), 'Sampling settings file is missing.'
print(json.dumps(json.loads(SAMPLING_SETTINGS_PATH.read_text()), indent=2))


## 6. Sample and resume the untuned baseline

The model loads once and batches prompts with eight draws per group. Each completed batch saves responses and a hash marker; a repeat verifies saved batches and generates only missing ones. An interrupted, unsealed batch is redone with its frozen seed. Settings/code/runtime mismatches stop reuse. Batch completion messages are work counters; this is not token-by-token progress. The subprocess exits on failure instead of polling a dead process.


In [ ]:
subprocess.run([sys.executable, '-m', 'pilot_eval', 'grpo-baseline',
                '--config', str(PREPARED_PATH), '--settings', str(SAMPLING_SETTINGS_PATH),
                '--name', BASELINE_NAME, '--output-root', str(ARTIFACT_ROOT)], cwd=REPO_DIR, check=True)
BASELINE_CONFIG = json.loads((ARTIFACT_ROOT / 'plans' / BASELINE_NAME / 'baseline.config.json').read_text())
BASELINE_DIR = ARTIFACT_ROOT / BASELINE_CONFIG['run_path']
BASELINE_SUMMARY = json.loads((BASELINE_DIR / 'results/results.json').read_text())
print(json.dumps({k:v for k,v in BASELINE_SUMMARY.items() if k != 'groups'}, indent=2))
print('Every draw:', BASELINE_DIR / 'results/responses.jsonl')
print('Per-group correct counts:', BASELINE_DIR / 'results/results.json')


## 7. Review the evidence before choosing the final cap

Read cap fraction, length percentiles, per-group correct counts and dead-group fraction. A dead group has all eight rewards equal (all correct or all incorrect), so it provides no within-group learning signal. Caps receive zero reward even if text contains an answer; this can encourage shorter responses.

If `p99_censored` is true, this run cannot justify the final cap. Preserve it and run an explicitly named extension with a larger provisional cap. Even an uncensored percentile does not guarantee later training responses will fit: the later protocol freeze must explicitly choose the final cap and cap-fraction monitoring policy. No training starts from this notebook stage yet.


## 8. Validate the GRPO implementation on CPU

This checks the training software before a scientific GPU run. It creates tiny random Qwen models locally; no pretrained model or dataset is downloaded. First it compares real TRL loss/LoRA gradients with an independent calculation and checks memory microbatch equivalence, padding, capped/dead groups and Adam momentum. Then it runs 20-step positive and reversed-sign learning controls at seeds 42 and 43.

The versioned learning gate compares steps 16–20 with the untouched policy's exact expected reward (50% in this binary task). Each sign must change reward by at least 20 points in each seed. Earlier failed fixtures remain preserved. The toy learning rate is distinct from the scientific rate.

Expected: `passed: true`, four passing learning controls, and saved configs, responses, per-step logs and initial/final adapters. Rerunning verifies and reuses sealed evidence. A failure stops the workflow; do not adjust a setting under the same run name. This does not authorize full training: GPU preflight and protocol freeze are still required.


In [ ]:
CONTROL_NAME = 'tiny-grpo-controls-v3'
subprocess.run([sys.executable, '-m', 'pilot_eval', 'grpo-controls',
                '--name', CONTROL_NAME, '--output-root', str(ARTIFACT_ROOT)],
               cwd=REPO_DIR, check=True)
CONTROL_DIR = ARTIFACT_ROOT / 'runs/diagnostics/pilot-4' / CONTROL_NAME
CONTROL_RESULTS = json.loads((CONTROL_DIR / 'results/results.json').read_text())
assert CONTROL_RESULTS['passed'] and CONTROL_RESULTS['algorithm_checks']['passed']
print('Algorithm checks:', CONTROL_RESULTS['algorithm_checks']['checks'])
for row in CONTROL_RESULTS['runs']:
    print('Seed:', row['seed'], 'sign:', row['sign'], 'baseline:', row['baseline_reward'],
          'late reward:', row['late_reward'], 'signed change:', row['signed_change'], 'passed:', row['passed'])
print('Saved evidence:', CONTROL_DIR)
